# Language-Model Fine-Tuning Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

You have a few hundred conversations from your own domain (this notebook ships with Filipino Q&A pairs) and an open, instruction-tuned language model that answers in the wrong language, register or format. Full fine-tuning would need tens of gigabytes of GPU memory and a multi-gigabyte copy of the weights for every variant you try.

With **QLoRA** you can adapt a 0.6B–4B model on a free Colab T4 in a few minutes, ship a **36 MB adapter** instead of a 1.4 GB model, and prove the adapter loads cleanly before anyone deploys it. That is what you will do here, end to end.

## What you will learn

**By the end of this notebook you will be able to:**
- **Normalize** instruction data from three common schemas into chat turns, and catch split leakage and over-length rows *before* training.
- **Mask the loss** to assistant tokens using the model's own chat template, and say exactly which tokens are and are not supervised.
- **Run 4-bit QLoRA** on a pinned base model and **read** the loss, perplexity and memory numbers it produces without over-claiming quality.
- **Export** a hash-manifested adapter bundle and **verify** a fresh reload from disk reproduces the adapted model.

Along the way you will see why pinning a model to an immutable revision matters, what a chat template actually renders, and why *optimization evidence* (loss went down) is not *task quality* (the model got good).

> approved pinned base → Hugging Face or local ZIP snapshot → sample or your own dataset → validation → baseline generation → QLoRA SFT → before/after generation → your own prompts → adapter export → fresh reload


## Prerequisites

**Required knowledge**
- Python: functions, dictionaries, list comprehensions.
- What a language model does at a high level (predicts the next token). You do not need transformer internals.

**Required tools**
- A Colab GPU runtime: *Runtime ▸ Change runtime type ▸ T4 GPU*. The defaults run in about **3–4 minutes** on a T4 (measured: 53 s of training, the rest is installing and downloading 1.4 GB of weights).
- Nothing else for the default model. Only the gated `llama-3.2-3b-instruct` needs a Hugging Face account, accepted terms, and an `HF_TOKEN` stored in Colab **Secrets** (the key icon in the left sidebar), with *Notebook access* enabled.

**How to use this notebook**
- Cells marked with a form on the right (`# @param`) are the knobs. Change them and re-run from that cell down.
- Run the cells in order the first time. Each section explains what the next cell does *before* it runs and what to look for in its output *after*.
- Every model here is a small **instruction-tuned** release, not a raw pre-trained checkpoint. You are adapting an assistant's language, tone and format on a small dataset, which is a different (and cheaper) job than turning a raw text completer into an assistant.


## 1. Set up the environment

Fine-tuning is memory-bound: activations, gradients and optimizer states all live on the GPU at once. Full 16-bit fine-tuning of even a 3B model needs more than 24 GB. **QLoRA** avoids that by freezing the base model in 4-bit precision and training only small low-rank adapter matrices, so the default run below peaks at about **1.5 GiB** of GPU memory.

The library versions are pinned. Chat-template rendering, 4-bit kernel selection and adapter serialization have all changed across releases, and a tutorial you cannot reproduce next month is not a tutorial.


In [ ]:
%pip -q install transformers==4.57.1 tokenizers==0.22.1 huggingface-hub==0.36.0 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.49.0 safetensors==0.8.0 "datasets>=3,<5"


Now import everything and confirm a GPU is visible. The printed line is your environment record; keep it if you report results.


In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import stat
import time
import zipfile
from pathlib import Path, PurePosixPath

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("No GPU visible. In Colab: Runtime > Change runtime type > T4 GPU, then re-run this cell.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Who wrote this notebook. Recorded into the exported artifact as provenance; it is not a sign-off.
AI_PROVENANCE = {
    "builders": [
        {"provider": "OpenAI", "product": "ChatGPT", "model": "GPT-5.6 Sol High", "role": "Builder"},
        {"provider": "Anthropic", "product": "Claude Code", "model": "Claude Fable 5.1", "role": "Reviewer and content revision"},
    ],
    "note": "Not independent reviewer sign-off",
}

print(f"Python {platform.python_version()} | torch {torch.__version__} | {GPU_NAME} | {GPU_VRAM_GB:.2f} GiB VRAM")


## 2. Choose a base model

The **base model** supplies the language ability and world knowledge; your data only nudges its style, language and format. Two things about this registry matter more than which model you pick:

1. **Every entry is pinned to a 40-character commit SHA.** A model name on the Hub (`Qwen/Qwen3-0.6B`) is a *branch*: its files can change under you. A SHA is immutable, so a run today and a run next year download byte-identical weights. Every load below also passes `trust_remote_code=False`, so no Python shipped with a model repository is ever executed.
2. **Every entry is already instruction-tuned.** Qwen3-0.6B, SmolLM2/3-Instruct, Granite-instruct, DeepSeek-R1-Distill, Qwen2.5-Coder-Instruct, danube3-chat and Llama-3.2-Instruct all answer questions out of the box. Fine-tuning them on 96 examples will shift *how* they answer, not *whether* they can. Expect subtle changes, and expect them to forget a little if you train hard.

The default, **`qwen3-0.6b`**, has two answer modes: *thinking* (it writes a `<think>…</think>` reasoning trace first) and *non-thinking* (it answers directly). This tutorial trains and evaluates in **non-thinking mode**, so the model learns to produce direct answers and you can compare before and after on equal terms. Section 5 shows exactly how that is done.

Other options in the form below: the Qwen3 1.7B/4B and Granite models carry measured minimum-VRAM gates from the repository's registry; the rest are unmeasured and print a note. `h2o-danube3-4b-chat`'s template does not accept a `system` role, so any system text is folded into the first user turn.

For **Llama 3.2 3B Instruct**, the repository is gated on Hugging Face. Accept the model terms, add a secret named `HF_TOKEN` in Colab Secrets, enable *Notebook access*, and keep `Pinned Hugging Face` as the source. The token stays in memory and is never written to the artifact.


In [ ]:
# One registry entry per line, on purpose: the repository's tests read these literally.
# Fields: model_id and revision (immutable Hub commit), license, min_vram_gb (measured QLoRA peak
# at 2048 tokens where known, else None), requires_hf_token (gated), dimer_zip (offline ZIP allowed).
TUTORIAL_REGISTRY={
"qwen3-0.6b":{"model_id":"Qwen/Qwen3-0.6B","revision":"c1899de289a04d12100db370d81485cdf75e47ca","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"smoke-ci/apache-2.0"},
"smollm3-3b":{"model_id":"HuggingFaceTB/SmolLM3-3B","revision":"a07cc9a04f16550a088caea529712d1d335b0ac1","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"tutorial-candidate/internal-only"},
"qwen3-1.7b":{"model_id":"Qwen/Qwen3-1.7B","revision":"70d244cc86ccca08cf5af4e1e306ecf908b1ad5e","license":"apache-2.0","min_vram_gb":9.3,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"qwen3-4b":{"model_id":"Qwen/Qwen3-4B","revision":"1cfa9a7208912126459214e8b04321603b3df60c","license":"apache-2.0","min_vram_gb":11.5,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"granite-4.1-3b":{"model_id":"ibm-granite/granite-4.1-3b","revision":"c0650403e44e78ec0262dab1c90914c65b196c4e","license":"apache-2.0","min_vram_gb":8.7,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"deepseek-r1-distill-qwen-1.5b":{"model_id":"deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B","revision":"ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562","license":"mit","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-reasoning/mit"},
"qwen2.5-coder-1.5b":{"model_id":"Qwen/Qwen2.5-Coder-1.5B-Instruct","revision":"2e1fd397ee46e1388853d2af2c993145b0f1098a","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-code/apache-2.0"},
"smollm2-1.7b":{"model_id":"HuggingFaceTB/SmolLM2-1.7B-Instruct","revision":"31b70e2e869a7173562077fd711b654946d38674","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-small/apache-2.0"},
"smollm2-360m":{"model_id":"HuggingFaceTB/SmolLM2-360M-Instruct","revision":"a10cc1512eabd3dde888204e902eca88bddb4951","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-small/apache-2.0"},
"granite-3.1-2b-instruct":{"model_id":"ibm-granite/granite-3.1-2b-instruct","revision":"bbc2aed595bd38bd770263dc3ab831db9794441d","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"enterprise-transparent/apache-2.0"},
"h2o-danube3-4b-chat":{"model_id":"h2oai/h2o-danube3-4b-chat","revision":"1e5c6fa6620f8bf078958069ab4581cd88e0202c","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"mobile-edge/apache-2.0"},
"llama-3.2-3b-instruct":{"model_id":"meta-llama/Llama-3.2-3B-Instruct","revision":"0cb88a4f764b7a12671c53f0838cd831a0843b95","license":"llama3.2","min_vram_gb":None,"requires_hf_token":True,"dimer_zip":False,"state":"credential-test-candidate"}}

BASE_MODEL_KEY = "qwen3-0.6b" # @param ["qwen3-0.6b","smollm3-3b","qwen3-1.7b","qwen3-4b","granite-4.1-3b","deepseek-r1-distill-qwen-1.5b","qwen2.5-coder-1.5b","smollm2-1.7b","smollm2-360m","granite-3.1-2b-instruct","h2o-danube3-4b-chat","llama-3.2-3b-instruct"]
MODEL_SOURCE = "Pinned Hugging Face" # @param ["Pinned Hugging Face","DIMER ZIP"]
TRAINING_METHOD = "qlora" # @param ["qlora"]
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
LORA_ALPHA = 16 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}

entry = TUTORIAL_REGISTRY[BASE_MODEL_KEY]
model_id = entry["model_id"]
revision = entry["revision"]
base_license = entry["license"]

if MODEL_SOURCE == "DIMER ZIP" and not entry["dimer_zip"]:
    raise RuntimeError(f"{BASE_MODEL_KEY} may not be redistributed as an offline ZIP; use 'Pinned Hugging Face'.")
if entry["min_vram_gb"] is None:
    print("Note: no measured VRAM profile for this model. This run records what it observes; it does not set a minimum.")
elif GPU_VRAM_GB < entry["min_vram_gb"]:
    raise RuntimeError(f"{BASE_MODEL_KEY} was measured to need {entry['min_vram_gb']} GiB at 2048 tokens; this GPU has {GPU_VRAM_GB:.1f} GiB.")

print(f"{BASE_MODEL_KEY}: {model_id} @ {revision[:12]} ({base_license}); source = {MODEL_SOURCE}")


## 3. Acquire the base model and its tokenizer

Two ways to get the weights:

- **`Pinned Hugging Face`** downloads the exact commit from the Hub (no credentials for public models; the `HF_TOKEN` preflight below runs only for gated ones and checks that the pinned SHA really exists before spending time on a 6 GB download).
- **`DIMER ZIP`** loads an offline snapshot produced by the repository's `scripts/fetch_weights.py --zip`. Before trusting anything inside it, the code checks that every archive member is a plain file with a safe relative path (no `..`, no absolute paths, no symlinks), that the archive carries a `dimer-base-manifest.json` naming *this* model and revision, that every listed file matches its recorded size and SHA-256, that no unlisted files were smuggled in, and that `.safetensors` weights are present. Skipping any one of those checks is how a tampered "model" gets executed.

The **tokenizer** is loaded here too, because the dataset step needs it to measure examples in tokens. The tokenizer turns text into integer ids and, through its **chat template**, wraps turns in the model's role markers (`<|im_start|>user … <|im_end|>` for the Qwen/ChatML family). `render_chat` below is the one function every later cell uses to render conversations, so training and inference can never drift apart in formatting.


In [ ]:
HF_TOKEN = None
if entry["requires_hf_token"]:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("Add HF_TOKEN in Colab Secrets, enable Notebook access, and accept the model terms on the Hub.") from exc
    hub_info = HfApi(token=HF_TOKEN).model_info(model_id, revision=revision)
    if hub_info.sha != revision:
        raise RuntimeError("The Hub resolved a different commit than the pinned revision.")
    print("Hugging Face credential and pinned-revision preflight passed.")

UPLOAD_DIMER_ZIP = False # @param {type:"boolean"}
DIMER_ZIP_PATH = "/content/dimer-base-model.zip" # @param {type:"string"}
EXPECTED_DIMER_ZIP_SHA256 = "" # @param {type:"string"}


def sha256_of_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_member_path(root, member_name):
    """Resolve an archive member name under root, refusing anything that could escape it."""
    if "\\" in member_name:
        raise ValueError(f"Unsafe ZIP path (backslash): {member_name!r}")
    relative = PurePosixPath(member_name)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe ZIP path: {member_name!r}")
    root = Path(root).resolve()
    target = (root / Path(*relative.parts)).resolve()
    if target != root and root not in target.parents:
        raise ValueError(f"ZIP member escapes the extraction root: {member_name!r}")
    return target


def extract_zip_safely(zip_path, root, size_limit_bytes):
    """Extract into root, refusing symlinks, path escapes and archives that expand past the limit."""
    root = Path(root).resolve()
    shutil.rmtree(root, ignore_errors=True)
    root.mkdir(parents=True)
    expanded = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if stat.S_ISLNK((info.external_attr >> 16) & 0xFFFF):
                raise ValueError("Symlinks are not allowed in the archive")
            expanded += info.file_size
            if expanded > size_limit_bytes:
                raise ValueError("Archive expands beyond the allowed size")
            target = safe_member_path(root, info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, open(target, "wb") as destination:
                shutil.copyfileobj(source, destination)
    return root


def verify_dimer_zip(zip_path):
    """Extract an offline base-model ZIP and verify it against its own manifest and this model's identity."""
    zip_path = Path(zip_path)
    archive_sha = sha256_of_file(zip_path)
    if EXPECTED_DIMER_ZIP_SHA256 and archive_sha.lower() != EXPECTED_DIMER_ZIP_SHA256.strip().lower():
        raise ValueError("DIMER ZIP SHA-256 does not match EXPECTED_DIMER_ZIP_SHA256")
    root = extract_zip_safely(zip_path, "/content/dimer-base-model", size_limit_bytes=20 * 1024**3)

    manifests = list(root.rglob("dimer-base-manifest.json"))
    if len(manifests) != 1:
        raise ValueError("Expected exactly one dimer-base-manifest.json in the archive")
    model_root = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())
    if manifest.get("format") != "dimer_hf_snapshot" or manifest.get("formatVersion") != 1:
        raise ValueError("Unsupported DIMER snapshot format")
    if (manifest.get("modelKey"), manifest.get("modelId"), manifest.get("revision")) != (BASE_MODEL_KEY, model_id, revision):
        raise ValueError("The archive describes a different model or revision than the one selected above")

    listed = set()
    listed_bytes = 0
    for record in manifest.get("files", []):
        file_path = safe_member_path(model_root, record["path"])
        if not file_path.is_file() or file_path.stat().st_size != record["bytes"] or sha256_of_file(file_path) != record["sha256"]:
            raise ValueError(f"DIMER file verification failed: {record['path']}")
        listed.add(record["path"])
        listed_bytes += record["bytes"]
    on_disk = {p.relative_to(model_root).as_posix() for p in model_root.rglob("*") if p.is_file() and p.name != "dimer-base-manifest.json"}
    if on_disk != listed or listed_bytes != manifest.get("totalBytes"):
        raise ValueError("Files on disk do not match the manifest (missing, extra, or wrong total size)")
    if not any(name.endswith(".safetensors") for name in listed):
        raise ValueError("The snapshot carries no .safetensors weights")
    return model_root, archive_sha


MODEL_LOAD_REF = model_id
BASE_MODEL_ACQUISITION = {"source": MODEL_SOURCE, "modelId": model_id, "revision": revision}
if MODEL_SOURCE == "DIMER ZIP":
    if UPLOAD_DIMER_ZIP:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one DIMER ZIP")
        Path(DIMER_ZIP_PATH).write_bytes(next(iter(uploaded.values())))
    MODEL_LOAD_REF, dimer_zip_sha = verify_dimer_zip(DIMER_ZIP_PATH)
    BASE_MODEL_ACQUISITION.update({"packageFormat": "dimer_hf_snapshot", "zipSha256": dimer_zip_sha})
    print("DIMER base-model package verified")

# The tokenizer is small; load it now so the dataset step can measure examples in tokens.
tokenizer_kwargs = {"trust_remote_code": False}
if MODEL_SOURCE == "Pinned Hugging Face":
    tokenizer_kwargs["revision"] = revision
    if HF_TOKEN:
        tokenizer_kwargs["token"] = HF_TOKEN
else:
    tokenizer_kwargs["local_files_only"] = True
tokenizer = AutoTokenizer.from_pretrained(MODEL_LOAD_REF, **tokenizer_kwargs)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
if not tokenizer.chat_template:
    raise ValueError("This tokenizer ships no chat template; this tutorial relies on one.")


def fold_system_into_first_user_turn(messages):
    """For templates that reject a system role: prepend the system text to the first user message."""
    system_text = messages[0]["content"].strip()
    if len(messages) > 1 and messages[1]["role"] == "user":
        merged = {"role": "user", "content": f"{system_text}\n\n{messages[1]['content'].strip()}"}
        return [merged, *messages[2:]]
    return [{"role": "user", "content": system_text}, *messages[1:]]


def render_chat(messages, add_generation_prompt=False):
    """Render a conversation with the model's own chat template.

    enable_thinking=False asks Qwen3-style templates for direct (non-thinking) answers; templates
    without that switch simply ignore it. Training and inference both go through this function,
    which is what keeps the two from drifting apart.
    """
    kwargs = {"tokenize": False, "add_generation_prompt": add_generation_prompt, "enable_thinking": False}
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except Exception as exc:
        if "System role not supported" in str(exc) and messages and messages[0]["role"] == "system":
            return tokenizer.apply_chat_template(fold_system_into_first_user_turn(messages), **kwargs)
        raise


sample_render = render_chat([{"role": "user", "content": "Kumusta?"}], add_generation_prompt=True)
print("A rendered prompt looks like this:\n" + sample_render)


## 4. Prepare the dataset

Supervised fine-tuning needs conversations, not rows. Whatever the source schema, every example is normalized to the same shape:

```json
{"messages": [
  {"role": "system",    "content": "You are a concise, domain-expert assistant."},
  {"role": "user",      "content": "Explain machine learning in one sentence."},
  {"role": "assistant", "content": "Machine learning is the science of training algorithms to learn patterns from data."}
]}
```

Three source schemas are recognized: chat records (`messages`), prompt/completion pairs, and Alpaca-style `instruction`/`input`/`output` records (the two shipped samples are Alpaca-shaped).

**Shipped samples.** `Sample: Filipino SFT` is a 513-row, AI-authored seed set (Apache-2.0) meant for exactly this kind of plumbing test. `Sample: Dolly` is `databricks-dolly-15k` (CC-BY-SA 3.0; adapters trained on it inherit the share-alike condition). Both are pinned to a dataset revision. For each, the notebook walks the rows in a deterministic hash order, keeps the first `SAMPLE_LIMIT` that fit within `MAX_SEQUENCE_LENGTH` **tokens**, and holds out one fifth of them as validation. This validation split is *manufactured from the sample*; it tells you whether training fit the format, not whether the model generalizes to your real task.

**Your own data.** Choose `Bring Your Own Dataset` and upload `train.jsonl` (plus optional `validation.jsonl`/`val.jsonl` and `test.jsonl`), or one ZIP containing them. Your rows are **not** pre-filtered: an over-length row stops the run in the next section rather than being silently truncated, because a truncated assistant turn teaches the model to stop mid-sentence.

**Hygiene checks that run every time**
- every example has at least one non-empty assistant turn;
- exact duplicates within a split are reported (not removed; that is your call);
- any example that appears in two splits halts the run: leakage makes validation numbers meaningless;
- a SHA-256 digest of the whole dataset is recorded into the artifact so a result can be tied to the exact data that produced it.


In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Sample: Dolly","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
MAX_TOTAL_TRAIN_TOKENS = 50_000_000

WORK_DIR = Path("/content/lm-sft")
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir()

ALLOWED_ROLES = {"system", "user", "assistant"}


def canonical(record):
    """Normalize one source record into {"messages": [...]} or raise if the schema is unknown."""
    if "messages" in record:
        messages = [{"role": m["role"], "content": str(m["content"])} for m in record["messages"]]
    elif "prompt" in record and ("completion" in record or "response" in record):
        answer = record.get("completion", record.get("response"))
        messages = [{"role": "user", "content": str(record["prompt"])}, {"role": "assistant", "content": str(answer)}]
    elif "instruction" in record and ("output" in record or "response" in record):
        context = record.get("input") or record.get("context")
        question = str(record["instruction"]) + (f"\n\n{context}" if context else "")
        answer = record.get("output", record.get("response"))
        messages = [{"role": "user", "content": question}, {"role": "assistant", "content": str(answer)}]
    else:
        raise ValueError("Unsupported SFT schema: expected messages, prompt/completion, or instruction/output")
    if any(m["role"] not in ALLOWED_ROLES for m in messages):
        raise ValueError("Invalid role in record")
    if not any(m["role"] == "assistant" and m["content"].strip() for m in messages):
        raise ValueError("Record has no non-empty assistant turn to learn from")
    return {"messages": messages}


def fingerprint(record):
    """Stable identity for a record: SHA-256 of its canonical JSON."""
    return hashlib.sha256(json.dumps(record, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()


def token_length(record):
    return len(tokenizer(render_chat(record["messages"]), add_special_tokens=False)["input_ids"])


if DATA_SOURCE.startswith("Sample"):
    if DATA_SOURCE == "Sample: Filipino SFT":
        dataset_id, dataset_revision, dataset_license = "jpaulpoliquit/ph-sft-ai-authored-v1", "8333699c6cc7296cc69cefc09def010851ded919", "apache-2.0"
    else:
        dataset_id, dataset_revision, dataset_license = "databricks/databricks-dolly-15k", "bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a", "cc-by-sa-3.0"
    all_rows = sorted((canonical(dict(row)) for row in load_dataset(dataset_id, revision=dataset_revision, split="train")), key=fingerprint)
    rows, skipped_long = [], 0
    for row in all_rows:
        if len(rows) == SAMPLE_LIMIT:
            break
        if token_length(row) <= MAX_SEQUENCE_LENGTH:
            rows.append(row)
        else:
            skipped_long += 1
    if skipped_long:
        print(f"Set aside {skipped_long} sample rows longer than {MAX_SEQUENCE_LENGTH} tokens while collecting {len(rows)}")
    validation_size = max(1, len(rows) // 5)
    SPLITS = {"train": rows[validation_size:], "validation": rows[:validation_size]}
    DATASET_PROVENANCE = {"source": dataset_id, "revision": dataset_revision, "license": dataset_license, "usage": "tutorial-training-not-benchmark"}
else:
    from google.colab import files
    uploaded = files.upload()
    byod_root = WORK_DIR / "byod"
    byod_root.mkdir()
    if len(uploaded) == 1 and next(iter(uploaded)).lower().endswith(".zip"):
        zip_path = WORK_DIR / "data.zip"
        zip_path.write_bytes(next(iter(uploaded.values())))
        extract_zip_safely(zip_path, byod_root, size_limit_bytes=2 * 1024**3)
    else:
        for name, payload in uploaded.items():
            (byod_root / Path(name).name).write_bytes(payload)

    def read_jsonl(path):
        return [canonical(json.loads(line)) for line in path.read_text().splitlines() if line.strip()]

    if not (byod_root / "train.jsonl").exists():
        raise ValueError("BYOD requires train.jsonl")
    if (byod_root / "validation.jsonl").exists() and (byod_root / "val.jsonl").exists():
        raise ValueError("Provide either validation.jsonl or val.jsonl, not both")
    SPLITS = {"train": read_jsonl(byod_root / "train.jsonl")}
    validation_file = byod_root / ("validation.jsonl" if (byod_root / "validation.jsonl").exists() else "val.jsonl")
    if validation_file.exists():
        SPLITS["validation"] = read_jsonl(validation_file)
    if (byod_root / "test.jsonl").exists():
        SPLITS["test"] = read_jsonl(byod_root / "test.jsonl")
    DATASET_PROVENANCE = {"source": "BYOD", "usage": "user-provided"}

# Hygiene: duplicates are reported, leakage across splits is fatal.
for split_name, records in SPLITS.items():
    if len(records) != len({fingerprint(r) for r in records}):
        print(f"Warning: exact duplicates inside the {split_name} split; none removed")
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    if left in SPLITS and right in SPLITS and {fingerprint(r) for r in SPLITS[left]} & {fingerprint(r) for r in SPLITS[right]}:
        raise ValueError(f"Split leakage: identical records in {left} and {right}")
if "validation" not in SPLITS:
    rows = SPLITS["train"]
    validation_size = max(1, len(rows) // 5)
    SPLITS = {**SPLITS, "train": rows[validation_size:], "validation": rows[:validation_size]}

DATASET_DIGEST = hashlib.sha256("".join(fingerprint(r) for name in sorted(SPLITS) for r in SPLITS[name]).encode()).hexdigest()
print({name: len(records) for name, records in SPLITS.items()}, "dataset digest", DATASET_DIGEST[:16])
print("First training example:")
print(json.dumps(SPLITS["train"][0], ensure_ascii=False, indent=2)[:600])


**What to look for.** With the defaults you should see `{'train': 96, 'validation': 24}` and a 16-character digest prefix. The digest is the identity of this exact dataset: if a colleague reports different numbers from "the same" data, compare digests first. With `Sample: Dolly`, the "set aside" line tells you how many rows were too long for the 512-token window; raise `MAX_SEQUENCE_LENGTH` if you want them, and expect memory to grow with it (Section 8).


## 5. Render chat turns and mask the loss

Training on a conversation means asking the model to predict every token in the rendered text. But you do not want it to learn to *write the user's questions*; you want it to learn to *answer them*. So the loss is computed only on assistant tokens. Everything else gets the label `-100`, which PyTorch's cross-entropy loss treats as "ignore" (`ignore_index`). This is **assistant-only loss masking**, and it is the single most common thing to get wrong in SFT.

Finding the assistant tokens is harder than it sounds, because the chat template wraps each turn in markers and, for Qwen3, inserts an empty `<think>\n\n</think>\n\n` scaffold before the final answer. The method used here:

1. Render the whole conversation with `render_chat` and tokenize it **with character offsets**, so every token knows which characters it covers.
2. For the **final assistant turn**, the span starts exactly where the generation prompt ends (`render_chat(messages_before_it, add_generation_prompt=True)`). That is the same prompt the model sees at inference, so what it is trained to produce is precisely what it will be asked to produce.
3. For **earlier assistant turns** in multi-turn data, the span is located by the turn's content.
4. Every span runs through the end-of-turn token (`<|im_end|>` for Qwen), so the model also learns *when to stop*.
5. A token whose characters straddle a span boundary stays masked; the model never trains on template scaffolding.

So for the default model, the supervised tokens are: **the answer text plus its end-of-turn marker**. Not the user turn, not the role markers, not the `<think>` scaffold (the template supplies that at inference). The next cell prints one example with the supervised part marked so you can see it.

Any example longer than `MAX_SEQUENCE_LENGTH` tokens stops the run here with `DATASET_SEQUENCE_TOO_LONG`. That is deliberate: silently truncating would cut answers mid-sentence.


In [ ]:
IGNORE_INDEX = -100


def encode(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def assistant_char_spans(messages, full_text):
    """Character ranges [start, end) of each assistant turn inside the rendered conversation."""
    end_of_turn = tokenizer.eos_token or "<|im_end|>"
    last_assistant = max(i for i, m in enumerate(messages) if m["role"] == "assistant")
    spans = []
    cursor = 0
    for index, message in enumerate(messages):
        content = message["content"].strip()
        if message["role"] != "assistant":
            found = full_text.find(content, cursor)
            if found != -1:
                cursor = found + len(content)
            continue
        start = -1
        if index == last_assistant:
            generation_prompt = render_chat(messages[:index], add_generation_prompt=True)
            if full_text.startswith(generation_prompt):
                start = len(generation_prompt)
        if start == -1:
            start = full_text.find(content, cursor)
            if start == -1:
                raise ValueError(f"Assistant content not found in rendered text: {content[:60]!r}")
        eos_at = full_text.find(end_of_turn, start)
        end = eos_at + len(end_of_turn) if eos_at != -1 else len(full_text)
        spans.append((start, end))
        cursor = end
    return spans


def build_masked_example(record):
    """Return (input_ids, labels) with labels = -100 everywhere except assistant turns."""
    messages = record["messages"]
    full_text = render_chat(messages)
    try:
        encoded = tokenizer(full_text, add_special_tokens=False, return_offsets_mapping=True)
        input_ids, offsets = encoded["input_ids"], encoded["offset_mapping"]
    except Exception:  # slow tokenizers cannot report offsets; fall back to prefix rendering
        input_ids, offsets = encode(full_text), None
    if len(input_ids) > MAX_SEQUENCE_LENGTH:
        raise ValueError("DATASET_SEQUENCE_TOO_LONG")

    labels = [IGNORE_INDEX] * len(input_ids)
    if offsets is not None:
        for start, end in assistant_char_spans(messages, full_text):
            for position, (char_start, char_end) in enumerate(offsets):
                inside_turn = start <= char_start and char_end <= end
                if inside_turn and char_start < char_end:
                    labels[position] = input_ids[position]
    else:
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            before = encode(render_chat(messages[:index], add_generation_prompt=True))
            through = encode(render_chat(messages[:index + 1]))
            if input_ids[:len(before)] != before or input_ids[:len(through)] != through:
                raise ValueError("Chat template rendering is not prefix-stable for this tokenizer")
            labels[len(before):len(through)] = input_ids[len(before):len(through)]

    if all(label == IGNORE_INDEX for label in labels):
        raise ValueError("No supervised tokens in example")
    return input_ids, labels


MASKED = {name: [build_masked_example(r) for r in records] for name, records in SPLITS.items()}
token_totals = {name: sum(len(ids) for ids, _ in examples) for name, examples in MASKED.items()}
supervised_totals = {name: sum(sum(label != IGNORE_INDEX for label in labels) for _, labels in examples) for name, examples in MASKED.items()}
if token_totals["train"] > MAX_TOTAL_TRAIN_TOKENS:
    raise ValueError("DATASET_TOKEN_BUDGET_EXCEEDED")
longest_train = max((len(ids) for ids, _ in MASKED["train"]), default=0)
if longest_train > 1024:
    print(f"Notice: the longest training example is {longest_train} tokens; sequences above ~1024 raise peak VRAM noticeably.")
print("tokens per split:", token_totals)
print("supervised (assistant) tokens per split:", supervised_totals)


Let's make the mask visible. The next cell decodes the first training example and wraps every supervised run of tokens in `⟦ ⟧`. Everything outside the brackets is context the model reads but is not graded on.


In [ ]:
def show_supervision(input_ids, labels):
    """Decode an example, marking supervised token runs with ⟦ ⟧."""
    pieces = []
    inside = False
    for token_id, label in zip(input_ids, labels):
        supervised = label != IGNORE_INDEX
        if supervised != inside:
            pieces.append("⟦" if supervised else "⟧")
            inside = supervised
        pieces.append(tokenizer.decode([token_id]))
    if inside:
        pieces.append("⟧")
    return "".join(pieces)


print(show_supervision(*MASKED["train"][0]))


## 6. Load the 4-bit base model and record a baseline

**Why 4-bit.** A 0.6B model in 16-bit is 1.2 GB of weights; in 4-bit **NormalFloat (`nf4`)** it is about 0.4 GB, and a 4B model drops from 8 GB to about 2.5 GB. `nf4` is a 16-level quantization grid shaped to the bell-curve distribution of trained weights, which is why it loses far less than a plain 4-bit integer grid would. **Double quantization** compresses the per-block scale factors as well. The frozen weights are dequantized on the fly to the compute dtype (bfloat16 on Ampere and newer, float16 on a T4) for each matrix multiply, and the attention uses PyTorch's memory-efficient `sdpa` kernels.

**Why a baseline first.** You cannot see what training changed unless you record what the model said *before*. The two probe prompts below are asked in non-thinking mode with greedy decoding (`do_sample=False`), so the comparison after training is deterministic. `generate` puts the model in eval mode (no dropout), turns the KV cache on, and restores whatever state it found, so the same helper is safe to call in the middle of training later.


In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)


def model_load_kwargs():
    kwargs = {
        "trust_remote_code": False,
        "dtype": compute_dtype,
        "quantization_config": quant_config,
        "device_map": {"": 0},
        "attn_implementation": "sdpa",
    }
    if MODEL_SOURCE == "Pinned Hugging Face":
        kwargs["revision"] = revision
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN
    else:
        kwargs["local_files_only"] = True
    return kwargs


base_model = AutoModelForCausalLM.from_pretrained(MODEL_LOAD_REF, **model_load_kwargs())
print(f"compute dtype: {compute_dtype} | GPU memory after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")


def generate(model, prompt, max_new_tokens=96):
    """Greedy, non-thinking, eval-mode generation; restores the model's training state afterwards."""
    was_training = model.training
    model.eval()
    previous_use_cache = getattr(model.config, "use_cache", True)
    model.config.use_cache = True
    prompt_text = render_chat([{"role": "user", "content": prompt}], add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    model.config.use_cache = previous_use_cache
    if was_training:
        model.train()
    return tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


PROMPTS = [
    "Ipaliwanag sa simpleng Filipino kung ano ang machine learning.",
    "Magbigay ng tatlong paraan para mabawasan ang basura sa opisina.",
]
BASELINE_OUTPUTS = [generate(base_model, prompt) for prompt in PROMPTS]
for prompt, answer in zip(PROMPTS, BASELINE_OUTPUTS):
    print(f"PROMPT: {prompt}\nBASE:   {answer}\n")


## 7. Attach LoRA adapters

**LoRA** keeps every original weight matrix $W_0$ frozen and learns a low-rank correction: $W = W_0 + \frac{\alpha}{r} B A$, where $A$ is $r \times d_{in}$ and $B$ is $d_{out} \times r$. With rank $r = 8$ that is a few thousand numbers per layer instead of millions. $B$ starts at zero, so at step 0 the adapted model *is* the base model; $\alpha / r$ scales how strongly the correction is applied (the defaults $r=8$, $\alpha=16$ give a scale of 2).

Adapters go on both the attention projections (`q/k/v/o_proj`) and the MLP projections (`gate/up/down_proj`). Attention-only LoRA is cheaper but adapts noticeably less; MLP layers hold most of a transformer's parameters.

`prepare_model_for_kbit_training` does the QLoRA housekeeping: it keeps layer norms and the output head in 32-bit for stable gradients through the 4-bit layers and turns on gradient checkpointing (recompute activations in the backward pass instead of storing them, trading compute for memory). The cell prints how many parameters are actually trainable; expect **under 1 %**.


In [ ]:
CANDIDATE_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
LORA_TARGETS = sorted({name.rsplit(".", 1)[-1] for name, _ in base_model.named_modules()} & CANDIDATE_TARGETS)
if not LORA_TARGETS:
    raise RuntimeError("No LoRA target modules found; this architecture names its projections differently")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGETS,
)
model = get_peft_model(prepare_model_for_kbit_training(base_model), lora_config)

trainable_params, all_params = model.get_nb_trainable_parameters()
print("LoRA targets:", LORA_TARGETS)
print(f"trainable parameters: {trainable_params:,} of {all_params:,} ({100 * trainable_params / all_params:.2f}%)")


## 8. Train

The loop below is deliberately plain PyTorch rather than a `Trainer`, so every moving part is visible:

- **One example per forward pass, optimizer step every two** (`GRAD_ACCUM = 2`). Batch size 1 keeps peak memory low and avoids padding; accumulating two examples before each step gives an effective batch of 2. Loss is scaled by the window size so the gradient matches a true batch.
- **AdamW at a constant learning rate** (`2e-4` is the usual LoRA starting point). There is no warmup or cosine schedule here; for one epoch on 96 examples a schedule changes little, and leaving it out keeps the loop readable. Add one when you train for real.
- **Losses are averaged per supervised token**, not per example, so a long answer counts for more than a short one, exactly as the optimizer sees it.
- **Validation loss** is measured in eval mode with gradients off.

Then the model is put in eval mode and the two probe prompts are asked again for the before/after table.


In [ ]:
GRAD_ACCUM = 2  # examples per optimizer step; batch size is 1, so this is the effective batch


def to_batch(example):
    input_ids, labels = example
    return {
        "input_ids": torch.tensor([input_ids], device="cuda"),
        "labels": torch.tensor([labels], device="cuda"),
        "attention_mask": torch.ones((1, len(input_ids)), dtype=torch.long, device="cuda"),
    }


def supervised_token_count(batch):
    return int((batch["labels"] != IGNORE_INDEX).sum())


def evaluate_loss(examples):
    """Mean cross-entropy per supervised token over a split, in eval mode with gradients off."""
    if not examples:
        return None
    was_training = model.training
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.inference_mode():
        for example in examples:
            batch = to_batch(example)
            count = supervised_token_count(batch)
            total_loss += model(**batch).loss.item() * count
            total_tokens += count
    if was_training:
        model.train()
    return total_loss / total_tokens if total_tokens else None


optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
random.seed(SEED)
torch.cuda.reset_peak_memory_stats()
started = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    examples = MASKED["train"][:]
    random.shuffle(examples)
    optimizer.zero_grad()
    epoch_loss, epoch_tokens, pending = 0.0, 0, 0
    for step, example in enumerate(examples, start=1):
        batch = to_batch(example)
        outputs = model(**batch)
        window = min(GRAD_ACCUM, len(examples) - (step - 1))  # the last window may hold a single example
        (outputs.loss / window).backward()
        count = supervised_token_count(batch)
        epoch_loss += outputs.loss.detach().item() * count
        epoch_tokens += count
        pending += 1
        if pending == window:
            optimizer.step()
            optimizer.zero_grad()
            pending = 0
    train_loss = epoch_loss / epoch_tokens
    validation_loss = evaluate_loss(MASKED["validation"])
    print(f"epoch {epoch}: train loss {train_loss:.4f} | validation loss {validation_loss:.4f}")

model.eval()
METRICS = {
    "trainLoss": train_loss,
    "validationLoss": validation_loss,
    "testLoss": evaluate_loss(MASKED.get("test", [])),
    "validationPerplexity": math.exp(validation_loss) if validation_loss is not None and validation_loss < 20 else None,
    "wallSeconds": time.time() - started,
    "peakGpuMemoryBytes": torch.cuda.max_memory_allocated(),
    "peakGpuMemoryGiB": torch.cuda.max_memory_allocated() / 1024**3,
}

ADAPTED_OUTPUTS = [generate(model, prompt) for prompt in PROMPTS]
display(pd.DataFrame({"prompt": PROMPTS, "base": BASELINE_OUTPUTS, "adapted": ADAPTED_OUTPUTS}))
print(json.dumps(METRICS, indent=2))


**What the numbers mean.** A reference run with the defaults on a Colab T4 gave train loss ≈ 3.22, validation loss ≈ 3.19, perplexity ≈ 24, 53 s, 1.52 GiB peak.

- **Validation loss slightly below training loss** after one epoch is normal here: dropout is active during training and the two splits come from the same distribution.
- **Perplexity 24** means the model was, on average, as unsure as choosing among 24 tokens at each answer position. On 24 held-out rows of the same style, that says the adapter is fitting the *format*. It says nothing about whether the answers are *correct*.
- These are **optimization** metrics. **Task quality** (does it answer your users well?) needs a held-out set from your real task, a rubric, and ideally human ratings. Do not ship on perplexity.
- **Try it:** set `EPOCHS = 3` and watch validation loss. On 96 rows it usually keeps falling through epoch 2 and turns up around epoch 3–4; that upturn is overfitting, and the lowest-validation-loss epoch is the one to keep.


## 9. Try your own prompts

The probe prompts above are the ones the model saw during evaluation. Real use means unseen instructions. The cell below asks two new questions and any prompt you type into `CUSTOM_PROMPT`, and shows the base and adapted answers side by side by switching the adapter off and on (`model.disable_adapter()`), which is the cleanest way to see what the adapter alone contributes.

Watch for: language and register (did it stay in Filipino?), whether it stops cleanly at the end of an answer, and repetition loops, which are the classic sign of a small model pushed too hard by greedy decoding.


In [ ]:
RUN_NEW_PROMPT_INFERENCE = True # @param {type:"boolean"}
CUSTOM_PROMPT = "" # @param {type:"string"}

NEW_PROMPTS = [
    "Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.",
    "Ipaliwanag ang pagkakaiba ng training data at evaluation data sa dalawang pangungusap.",
]
if CUSTOM_PROMPT.strip():
    NEW_PROMPTS.append(CUSTOM_PROMPT.strip())


def base_and_adapted(prompt):
    with model.disable_adapter():
        base_answer = generate(model, prompt)
    return base_answer, generate(model, prompt)


if RUN_NEW_PROMPT_INFERENCE:
    pairs = [base_and_adapted(prompt) for prompt in NEW_PROMPTS]
    display(pd.DataFrame({"prompt": NEW_PROMPTS, "base": [b for b, _ in pairs], "adapted": [a for _, a in pairs]}))


## 10. Export the adapter and prove it reloads

The deliverable is the **adapter bundle**, not a copy of the base model. It contains the LoRA matrices (`adapter_model.safetensors`, ~20 MB for the default), the adapter config, the tokenizer with its chat template (so inference renders prompts exactly as training did), `metrics.json`, `provenance.json` (base model id and pinned revision, dataset digest and license, hyperparameters, runtime, authorship), and an `artifact-manifest.json` listing every file with its size and SHA-256. The companion inference notebook refuses any bundle whose files do not match that manifest.

Then the notebook proves the bundle is usable **from disk**: it deletes the in-memory model, loads the base model again, attaches the saved adapter with `PeftModel.from_pretrained(..., is_trainable=False)`, and checks three things:

1. the reloaded adapter's `B` matrices are not all zero (an untrained or mis-saved adapter would be indistinguishable from the base model);
2. with the adapter switched **off**, the reloaded model answers the first probe prompt differently than with it **on** (so the deltas are really being applied, not silently ignored);
3. it generates text for a fresh prompt.

The cell also prints the opening of the in-memory adapted answer next to the reloaded one. They normally agree for the first several tokens and then drift: the training model keeps its layer norms and output head in 32-bit (`prepare_model_for_kbit_training` did that), the reloaded one uses the base checkpoint's 16-bit versions, and greedy decoding amplifies any tiny logit difference once it flips a single token. That drift is why the check above compares adapter-on with adapter-off on the *same* model rather than demanding byte-identical text across two models.

Only then is the bundle zipped and offered for download.


In [ ]:
STAGING_DIR = Path("/content/dimer-lm-adapter.staging")
ADAPTER_DIR = Path("/content/dimer-lm-adapter")
ARTIFACT_ZIP = Path("/content/dimer-language-model-adapter.zip")
shutil.rmtree(STAGING_DIR, ignore_errors=True)
shutil.rmtree(ADAPTER_DIR, ignore_errors=True)
STAGING_DIR.mkdir()

model.save_pretrained(STAGING_DIR, safe_serialization=True)
tokenizer.save_pretrained(STAGING_DIR / "tokenizer")

PROVENANCE = {
    "artifactFormat": "peft_adapter",
    "artifactFormatVersion": 1,
    "baseModel": model_id,
    "baseModelRevision": revision,
    "baseModelLicense": base_license,
    "modelKey": BASE_MODEL_KEY,
    "trustRemoteCode": False,
    "requiresHfToken": bool(entry["requires_hf_token"]),
    "dimerZipAllowed": bool(entry["dimer_zip"]),
    "baseModelAcquisition": BASE_MODEL_ACQUISITION,
    "datasetDigest": DATASET_DIGEST,
    "dataset": DATASET_PROVENANCE,
    "training": {"method": TRAINING_METHOD, "epochs": EPOCHS, "learningRate": LEARNING_RATE, "loraRank": LORA_RANK, "loraAlpha": LORA_ALPHA, "targetModules": LORA_TARGETS, "maxSequenceLength": MAX_SEQUENCE_LENGTH},
    "runtime": {"python": platform.python_version(), "torch": torch.__version__, "gpu": GPU_NAME, "gpuVramGiB": GPU_VRAM_GB},
    "aiProvenance": AI_PROVENANCE,
}
(STAGING_DIR / "metrics.json").write_text(json.dumps(METRICS, indent=2))
(STAGING_DIR / "provenance.json").write_text(json.dumps(PROVENANCE, indent=2))
(STAGING_DIR / "MODEL_CARD.md").write_text(
    f"# PEFT adapter for {model_id}\n\nBase revision: `{revision}`. Dataset digest: `{DATASET_DIGEST}`.\n"
    "Optimization metrics in metrics.json are not task-quality evidence.\n"
)

manifest_records = [
    {"path": path.relative_to(STAGING_DIR).as_posix(), "bytes": path.stat().st_size, "sha256": sha256_of_file(path)}
    for path in sorted(STAGING_DIR.rglob("*"))
    if path.is_file() and path.name != "artifact-manifest.json"
]
(STAGING_DIR / "artifact-manifest.json").write_text(json.dumps(
    {"format": "peft_adapter", "formatVersion": 1, "files": manifest_records, "totalBytes": sum(r["bytes"] for r in manifest_records)}, indent=2))
for record in manifest_records:
    if sha256_of_file(STAGING_DIR / record["path"]) != record["sha256"]:
        raise RuntimeError("artifact-manifest.json verification failed")

# Fresh base + adapter reload, from disk, before publication.
REPLAY_TOKENS = 16
expected_opening = generate(model, PROMPTS[0], max_new_tokens=REPLAY_TOKENS)
del model, base_model
gc.collect()
torch.cuda.empty_cache()

reloaded_base = AutoModelForCausalLM.from_pretrained(MODEL_LOAD_REF, **model_load_kwargs())
tokenizer = AutoTokenizer.from_pretrained(STAGING_DIR / "tokenizer", local_files_only=True, trust_remote_code=False)
reloaded_model = PeftModel.from_pretrained(reloaded_base, STAGING_DIR, is_trainable=False)

lora_b_matrices = [param for name, param in reloaded_model.named_parameters() if "lora_b" in name.lower()]
if not lora_b_matrices or max(p.abs().max().item() for p in lora_b_matrices) == 0:
    raise RuntimeError("Reloaded adapter weights are zero or missing")
reloaded_opening = generate(reloaded_model, PROMPTS[0], max_new_tokens=REPLAY_TOKENS)
with reloaded_model.disable_adapter():
    adapter_off_opening = generate(reloaded_model, PROMPTS[0], max_new_tokens=REPLAY_TOKENS)
if reloaded_opening == adapter_off_opening:
    raise RuntimeError("Reloaded adapter has no effect: adapter-on and adapter-off answers are identical")
smoke = generate(reloaded_model, "Kumusta! Sagutin sa isang maikling pangungusap.", max_new_tokens=32)
if not smoke:
    raise RuntimeError("Fresh reload generated nothing")
print("✓ Fresh base + adapter reload: adapter weights present and active")
print("  in-memory adapted opening:", expected_opening)
print("  reloaded  adapted opening:", reloaded_opening)
print("  reloaded  adapter-off    :", adapter_off_opening)
print("  fresh prompt ->", smoke)

os.replace(STAGING_DIR, ADAPTER_DIR)
with zipfile.ZipFile(ARTIFACT_ZIP, "w", zipfile.ZIP_STORED) as archive:
    for path in ADAPTER_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(ADAPTER_DIR).as_posix())
print(f"Artifact: {ARTIFACT_ZIP} ({ARTIFACT_ZIP.stat().st_size / 1024**2:.1f} MB) SHA-256 {sha256_of_file(ARTIFACT_ZIP)}")

from google.colab import files
files.download(str(ARTIFACT_ZIP))


## What a successful run proves

If every cell ran, the **exact path you selected** has produced evidence for:
1. pinned base-model acquisition with integrity checks (and, for a DIMER ZIP, manifest verification);
2. dataset normalization, duplicate reporting and split-leakage checking, with a recorded digest;
3. chat-template rendering with assistant-only loss masking, shown token by token;
4. 4-bit QLoRA training with per-token loss, validation loss, perplexity, time and peak memory;
5. before/after and novel-prompt comparisons with the adapter switched off and on;
6. a manifested adapter bundle that reloads from disk and reproduces the adapted model's answer.

It does **not** prove the adapter is good at your task. That needs your own held-out evaluation.

### Recap against the learning objectives
- *Normalize and validate data*: Section 4 (three schemas, hygiene checks, token-length gate, digest).
- *Mask the loss and say what is supervised*: Section 5 (answer text + end-of-turn marker; not the user turn, role markers or think scaffold).
- *Run QLoRA and read the numbers*: Sections 6–8 (nf4, LoRA rank/alpha, accumulation, loss vs perplexity vs quality).
- *Export and verify*: Section 10 (manifest, reload, answer reproduction).

### Next experiments, in the order they teach the most
1. `EPOCHS = 3`: find the epoch where validation loss turns up.
2. `LORA_RANK = 16, LORA_ALPHA = 32`: more capacity; compare validation loss and the novel-prompt answers.
3. `Sample: Dolly`: English data; watch the adapted model's Filipino answers drift toward English (that is forgetting, on a small scale).
4. `Bring Your Own Dataset` with 200–500 rows from your domain and a real held-out test set.
5. `MAX_SEQUENCE_LENGTH = 1024`: memory roughly follows sequence length squared through the attention scores; watch `peakGpuMemoryGiB`.

### Troubleshooting
| Symptom | Cause | What to do |
|---|---|---|
| `DATASET_SEQUENCE_TOO_LONG` | a BYOD row renders to more than `MAX_SEQUENCE_LENGTH` tokens | raise the limit or shorten the row; the notebook never truncates silently |
| `Split leakage …` | the same record appears in two of your splits | deduplicate across files before uploading |
| `CUDA out of memory` at Section 8 | long sequences, or a larger model than this GPU fits | lower `MAX_SEQUENCE_LENGTH`, pick a smaller model, or a bigger GPU |
| `401` / `GatedRepoError` when loading Llama | terms not accepted, or `HF_TOKEN` not shared with this notebook | accept on the Hub; Secrets ▸ enable Notebook access |
| repetition loops in answers | greedy decoding on a small model | expected for smoke tests; use sampling for real use (the inference notebook shows the settings) |

### Deploying the adapter
- **Attach at load time** with `peft` and `transformers`, exactly as the companion [Artifact Inference](language_model_artifact_inference_colab.ipynb) notebook does. Simplest, and what this bundle is built for.
- **Multi-adapter serving** (vLLM, SGLang, TGI) keeps one base model in memory and routes requests to many adapters; hand the server the bundle's `adapter_model.safetensors` and `adapter_config.json`.
- **Merging into the weights** (`merge_and_unload()`) is possible but *not* on the 4-bit model you trained on: merging into quantized weights loses precision (the pitfall this notebook warns about). Load the base model in 16-bit on a GPU that fits it, attach the adapter, merge, `save_pretrained`, and convert (for example to GGUF) for llama.cpp or Ollama.

### Licenses and provenance
The adapter inherits obligations from both its base model (recorded as `baseModelLicense`) and its training data (`dataset.license`); Dolly's CC-BY-SA, for instance, is share-alike. Both are written into `provenance.json`, together with who and what produced this notebook (`aiProvenance`), so a downstream reader can audit the bundle without this notebook.
